In [1]:
from pathlib import Path
import duckdb
import polars as pl

In [2]:
# Setup Path: Mundur satu langkah dari folder 'notebooks' menggunakan .parent
PROJECT_ROOT = Path.cwd().resolve().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "chicago_crimes.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "chicago_crimes_clean.parquet"

# Menggunakan LazyFrame agar hemat RAM untuk memproses 8,1 juta baris
lf = pl.scan_csv(
    DATA_PATH,
    try_parse_dates=True,
    ignore_errors=True,
    infer_schema_length=10000
)

# Cleaning 
rename_map = {col: col.strip().lower().replace(" ", "_") for col in lf.collect_schema().names()}
lf_std = lf.rename(rename_map)

print("Succsess! Total kolom:", len(rename_map))

Succsess! Total kolom: 22


In [3]:
con = duckdb.connect()

# Menjalankan profiling langsung dari file mentah CSV
profil = con.execute(f"SUMMARIZE SELECT * FROM read_csv_auto('{DATA_PATH}', ignore_errors=true)").pl()
profil

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""ID""","""BIGINT""","""634""","""13525542""",8692979,"""7272202.810450484""","""3642948.1631511883""","""3901077""","""7287280""","""10535114""",8104658,0.00
"""Case Number""","""VARCHAR""","""01G050460""","""ZZZ199957""",9812079,null,null,null,null,null,8104658,0.00
"""Date""","""TIMESTAMP""","""2001-01-01 00:00:00""","""2024-07-04 00:00:00""",3775371,"""2010-12-22 13:37:17.028808""",null,"""2005-04-03 14:11:36.706555""","""2009-12-21 10:34:39.946107""","""2016-05-16 01:15:19.433226""",8104658,0.00
"""Block""","""VARCHAR""","""0000X E 100 PL""","""XX UNKNOWN""",54224,null,null,null,null,null,8104658,0.00
"""IUCR""","""VARCHAR""","""0110""","""9901""",484,null,null,null,null,null,8104658,0.00
…,…,…,…,…,…,…,…,…,…,…,…
"""Year""","""BIGINT""","""2001""","""2024""",26,"""2010.4736085100692""","""6.67017533427587""","""2005""","""2010""","""2016""",8104658,0.00
"""Updated On""","""TIMESTAMP""","""2006-03-31 22:03:38""","""2024-07-11 15:45:11""",6081,"""2018-08-22 23:40:18.115331""",null,"""2018-02-10 15:50:01""","""2018-02-28 15:56:25""","""2018-02-28 20:13:04.110543""",8104658,0.00
"""Latitude""","""DOUBLE""","""36.619446395""","""42.022910333""",864066,"""41.84237384933097""","""0.08933712309827614""","""41.768872631153044""","""41.85637718805103""","""41.90704123884905""",8104658,1.11


In [4]:
# Cek sebaran null
print(lf_std.null_count().collect())

# Parsing format tanggal dan menghapus missing values pada kolom krusial
lf_missing_handled = (
    lf_std
    .with_columns(
        pl.col("date").str.to_datetime("%m/%d/%Y %I:%M:%S %p", strict=False).alias("datetime")
    )
    .drop_nulls(subset=["datetime", "latitude", "longitude", "primary_type"])
)

shape: (1, 22)
┌─────┬─────────────┬──────┬───────┬───┬────────────┬──────────┬───────────┬──────────┐
│ id  ┆ case_number ┆ date ┆ block ┆ … ┆ updated_on ┆ latitude ┆ longitude ┆ location │
│ --- ┆ ---         ┆ ---  ┆ ---   ┆   ┆ ---        ┆ ---      ┆ ---       ┆ ---      │
│ u32 ┆ u32         ┆ u32  ┆ u32   ┆   ┆ u32        ┆ u32      ┆ u32       ┆ u32      │
╞═════╪═════════════╪══════╪═══════╪═══╪════════════╪══════════╪═══════════╪══════════╡
│ 0   ┆ 0           ┆ 0    ┆ 0     ┆ … ┆ 0          ┆ 89642    ┆ 89642     ┆ 89642    │
└─────┴─────────────┴──────┴───────┴───┴────────────┴──────────┴───────────┴──────────┘


In [6]:
# Batas Geografis Chicago (Domain Knowledge):
# Latitude : 41.60 s/d 42.05
# Longitude: -87.95 s/d -87.50

lf_outlier_handled = lf_missing_handled.filter(
    pl.col("latitude").is_between(41.60, 42.05) &
    pl.col("longitude").is_between(-87.95, -87.50) &
    (pl.col("datetime").dt.year() >= 2001) &
    (pl.col("datetime").dt.year() <= 2026)
)

In [7]:
lf_dedup = lf_outlier_handled.unique(subset=["id"], keep="first")

In [8]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Mengeksekusi seluruh pipeline secara streaming dan menyimpan hasilnya
lf_dedup.sink_parquet(OUTPUT_PATH)

print(f"Data bersih berhasil disimpan dalam format Parquet di: {OUTPUT_PATH.name}")

Data bersih berhasil disimpan dalam format Parquet di: chicago_crimes_clean.parquet


In [9]:
# Menguji file Parquet hasil pembersihan menggunakan DuckDB
validasi = con.execute(f"""
    SELECT
        COUNT(*) AS total_baris_bersih,
        SUM(CASE WHEN latitude IS NULL THEN 1 ELSE 0 END) AS sisa_null_lat,
        SUM(CASE WHEN datetime IS NULL THEN 1 ELSE 0 END) AS sisa_null_date,
        SUM(CASE WHEN latitude < 41.60 OR latitude > 42.05 THEN 1 ELSE 0 END) AS sisa_outlier_lat,
        SUM(CASE WHEN longitude < -87.95 OR longitude > -87.50 THEN 1 ELSE 0 END) AS sisa_outlier_lon
    FROM read_parquet('{OUTPUT_PATH}')
""").pl()

validasi

total_baris_bersih,sisa_null_lat,sisa_null_date,sisa_outlier_lat,sisa_outlier_lon
i64,"decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]"
8014867,0,0,0,0
